# Comprehensive EDA Workspace

This notebook provides an in-depth Exploratory Data Analysis covering datatypes, null-value checks, summary statistics (for outliers), cardinality checks (for categoricals), and visual distributions.

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, count, when, isnan, countDistinct
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = SparkSession.builder.appName("EDA").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Started")

## 1. Load the Data


In [ ]:
df_loans = spark.read.csv("data/lms_loan_daily.csv", header=True, inferSchema=True)
df_attr = spark.read.csv("data/features_attributes.csv", header=True, inferSchema=True)
df_fin = spark.read.csv("data/features_financials.csv", header=True, inferSchema=True)
df_click = spark.read.csv("data/feature_clickstream.csv", header=True, inferSchema=True)

print(f"Loans Count: {df_loans.count()}")
print(f"Attributes Count: {df_attr.count()}")
print(f"Financials Count: {df_fin.count()}")
print(f"Clickstream Count: {df_click.count()}")

## 2. Check Datatypes (Schema)
Review these schemas to ensure dates are parsed as Dates, numbers as Int/Double, and categories as Strings.

In [ ]:
print("--- Loans Schema ---")
df_loans.printSchema()
print("\n--- Attributes Schema ---")
df_attr.printSchema()
print("\n--- Financials Schema ---")
df_fin.printSchema()
print("\n--- Clickstream Schema ---")
df_click.printSchema()

## 3. Data Cleanness: Check for Missing Values (NULLs)


In [ ]:
def count_nulls(df):
    exprs = []
    for c, t in df.dtypes:
        if t in ('double', 'float'):
            # isnan only works on floating point types
            exprs.append(_sum(when(col(c).isNull() | isnan(col(c)), 1).otherwise(0)).alias(c))
        else:
            exprs.append(_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c))
    return df.select(exprs)

print("Nulls in Loans:")
count_nulls(df_loans).show()
print("Nulls in Attributes:")
count_nulls(df_attr).show()
print("Nulls in Financials:")
count_nulls(df_fin).show()
print("Nulls in Clickstream:")
count_nulls(df_click).show()

## 4. Summary Statistics (Outlier Detection)
We use `.describe()` on numeric columns to see min, max, mean, and stddev to identify potential outliers.

In [ ]:
def get_numeric_cols(df):
    return [c for c, t in df.dtypes if t in ('int', 'double', 'float', 'bigint')]

print("--- Loans Summary ---")
df_loans.select(get_numeric_cols(df_loans)).describe().show()

print("--- Attributes Summary ---")
df_attr.select(get_numeric_cols(df_attr)).describe().show()

print("--- Financials Summary ---")
df_fin.select(get_numeric_cols(df_fin)).describe().show()


## 5. Cardinality Check (Categorical Columns)
Checking how many unique values exist in string/categorical columns.

In [ ]:
def get_string_cols(df):
    return [c for c, t in df.dtypes if t == 'string']

def check_cardinality(df, name):
    string_cols = get_string_cols(df)
    if not string_cols: return
    print(f"--- Cardinality for {name} ---")
    df.select([countDistinct(c).alias(c) for c in string_cols]).show()

check_cardinality(df_loans, "Loans")
check_cardinality(df_attr, "Attributes")
check_cardinality(df_fin, "Financials")
check_cardinality(df_click, "Clickstream")

## 6. Visualizations: Spread and Outliers
We sample the data to pandas for plotting.

In [ ]:
pd_attr = df_attr.limit(10000).toPandas()
pd_fin = df_fin.limit(10000).toPandas()

plt.style.use('ggplot')

# Boxplots for Numeric Spread
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
if 'age' in pd_attr.columns:
    sns.boxplot(x=pd_attr['age'], ax=axes[0], color='skyblue')
    axes[0].set_title('Boxplot of Age (Detecting Outliers)')

if 'income' in pd_fin.columns:
    sns.boxplot(x=pd_fin['income'], ax=axes[1], color='salmon')
    axes[1].set_title('Boxplot of Income (Detecting Outliers)')
plt.tight_layout()
plt.show()

In [ ]:
# Distributions for Categoricals
if 'marital_status' in pd_attr.columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=pd_attr, x='marital_status', palette='viridis')
    plt.title('Marital Status Distribution')
    plt.show()